In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import warnings

from arch import arch_model
from tqdm import tqdm #progress bar
from statsmodels.tsa.arima.model import ARIMA
import pickle

warnings.filterwarnings("ignore")
from hurst import compute_Hc

In [ ]:
tickers=[
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS", "HINDUNILVR.NS",
    "ITC.NS", "KOTAKBANK.NS", "LT.NS", "SBIN.NS", "AXISBANK.NS", "BAJFINANCE.NS",
    "BHARTIARTL.NS", "ASIANPAINT.NS", "HCLTECH.NS", "MARUTI.NS", "SUNPHARMA.NS", "TITAN.NS",
    "ULTRACEMCO.NS", "WIPRO.NS", "TECHM.NS", "NESTLEIND.NS", "POWERGRID.NS", "NTPC.NS",
    "JSWSTEEL.NS", "TATASTEEL.NS", "BAJAJFINSV.NS", "ADANIENT.NS", "ADANIPORTS.NS", "HINDALCO.NS",
    "ONGC.NS", "COALINDIA.NS", "EICHERMOT.NS", "HDFCLIFE.NS", "SBILIFE.NS", "BRITANNIA.NS",
    "DIVISLAB.NS", "CIPLA.NS", "GRASIM.NS", "BPCL.NS", "IOC.NS", "HEROMOTOCO.NS",
    "BAJAJ-AUTO.NS", "SHREECEM.NS", "DRREDDY.NS", "M&M.NS", "INDUSINDBK.NS", "APOLLOHOSP.NS"
]

In [ ]:
start_date='2022-03-01'
end_date='2024-01-31'

avg_forecasted_vol=[]

data=yf.download(tickers,start=start_date,end=end_date,interval="1d")
returns=data["Close"].pct_change().dropna()
returns=returns*100
display(returns)

[*********************100%***********************]  48 of 48 completed


Ticker,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BHARTIARTL.NS,BPCL.NS,...,SBILIFE.NS,SBIN.NS,SHREECEM.NS,SUNPHARMA.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2022-03-03,0.121818,-0.506825,1.163272,-5.184968,-1.679432,-2.229932,-2.173430,-1.217872,-0.326282,1.256935,...,-3.554383,-1.444380,-4.619654,0.000000,1.081940,-0.049358,2.316506,-0.709716,-6.542062,2.584652
2022-03-04,-1.752037,-0.926838,-3.468005,-4.657194,-3.070565,-1.668124,-2.361821,-3.118552,-2.730454,0.562932,...,-2.214932,-1.166023,-1.212158,1.065904,-2.010285,-0.565674,1.889030,-5.176254,1.163733,1.009567
2022-03-07,-3.154802,-3.192192,-1.746266,-1.084660,-6.636370,-0.229799,-6.260917,-6.315476,3.319570,-2.971157,...,-2.690702,-4.686655,-2.483340,-0.843738,1.158877,-1.119348,-2.029435,-2.101598,-5.821203,-0.599690
2022-03-08,2.400823,1.630293,1.241333,0.551964,0.426941,2.548964,0.539199,0.601619,1.125256,1.065095,...,0.296439,0.000000,2.698931,3.932418,-1.733881,3.301399,2.657802,-0.772071,2.375359,2.728003
2022-03-09,3.430949,3.665518,-0.239300,5.564732,0.566823,0.660217,3.906383,5.041523,1.244509,0.307392,...,-0.654109,2.589144,-2.741786,1.549708,-1.114603,0.906946,2.927609,2.631527,3.152509,-0.331945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-01-23,-0.655085,-1.331948,0.969706,-3.510906,-2.445039,-0.569604,-1.124647,-3.393853,2.933333,-3.721806,...,-4.518953,-3.608137,3.167978,3.197061,-3.091250,-2.150625,-1.672771,-1.861923,-2.602618,-3.123388
2024-01-24,0.238218,-1.451050,0.968510,-1.808694,-2.763362,1.648140,2.423379,-0.217798,2.759068,3.045031,...,2.548162,2.181458,0.514297,0.275703,3.881621,-0.426356,3.021978,0.754094,1.622425,1.951379
2024-01-25,-0.339254,2.293414,-0.716194,-1.675304,-1.595620,5.332837,0.573777,0.428046,-2.470688,-0.576345,...,-2.102011,-0.897612,1.134976,-0.998474,-1.035889,-0.819933,-6.101054,0.062361,-0.211708,-1.683926


In [ ]:
def arima_forecast_series(returns, tickers, window_size=90, forecast_horizon=1, order=(1,0,1), eps=1e-8):
    vol_forecast_dict = {}
    avg_forecasted_vol_list = []

    for ticker in tqdm(tickers, desc="ARIMA rolling forecasts"):
        if ticker not in returns.columns:
            vol_forecast_dict[ticker] = pd.Series(dtype=float)
            avg_forecasted_vol_list.append(np.nan)
            continue
        sq_ret = (returns[ticker].dropna()) ** 2
        rolling_vols = []
        rolling_dates = []

        if len(sq_ret) < window_size + 1:
            vol_forecast_dict[ticker] = pd.Series(dtype=float)
            avg_forecasted_vol_list.append(np.nan)
            continue

        for i in range(window_size, len(sq_ret)):
            train_window = sq_ret.iloc[i - window_size:i]

            if train_window.isnull().any() or train_window.std() < 1e-12:
                continue
            y = np.log(train_window + eps)

            try:
                model = ARIMA(y, order=order)
                res = model.fit(method_kwargs={"warn_convergence": False})
                fc = res.forecast(steps=forecast_horizon)
                pred_log_var = float(fc[0])

                pred_var = np.exp(pred_log_var)         # variance in %^2
                pred_vol = np.sqrt(pred_var) / 100.0    # convert to proportion

                if not np.isnan(pred_vol) and pred_vol < 10:
                    rolling_vols.append(pred_vol)
                    rolling_dates.append(sq_ret.index[i])

            except Exception:
                continue

        vol_series = pd.Series(rolling_vols, index=rolling_dates)
        vol_forecast_dict[ticker] = vol_series

        avg_vol = np.nan if not rolling_vols else float(np.mean(rolling_vols))
        avg_forecasted_vol_list.append(avg_vol)

        print(f"{ticker} avg forecasted vol (daily) = {avg_vol}")

    vol_forecast_df = pd.DataFrame(vol_forecast_dict)
    avg_forecasted_vol_df = pd.DataFrame(avg_forecasted_vol_list, index=tickers, columns=['avg_vol'])

    return vol_forecast_df, avg_forecasted_vol_df
def garch_forecast_series(returns_df, tickers, window=90, p=1, q=1, dist='t', forecast_horizon=1):
    vol_forecast_dict = {}
    avg_forecasted_vol_list = []

    for ticker in tqdm(tickers):
        rolling_volatility = []
        dates = []
        ret_series = returns_df[ticker]

        for i in range(window, len(ret_series)):
            train_data = ret_series.iloc[i - window:i]

            if train_data.isna().any() or train_data.std() < 1e-6:
                continue

            try:
                model = arch_model(train_data * 100, vol='Garch', p=p, q=q, dist=dist, mean='Constant')
                result = model.fit(disp='off', show_warning=False)
                forecast = result.forecast(horizon=forecast_horizon, reindex=False)
                var = forecast.variance.values[-1, 0] / (100**2)

                vol_annual = np.sqrt(var * 252)

                if not np.isnan(vol_annual) and vol_annual < 100:
                    rolling_volatility.append(vol_annual)
                    dates.append(ret_series.index[i])
            except Exception:
                continue

        vol_series = pd.Series(rolling_volatility, index=dates)
        vol_forecast_dict[ticker] = vol_series

        avg_forecasted_vol = np.mean(rolling_volatility) if rolling_volatility else np.nan
        avg_forecasted_vol_list.append(avg_forecasted_vol)
        print(f"avg_forecasted_vol of {ticker} = {avg_forecasted_vol}")

    vol_forecast_df = pd.DataFrame(vol_forecast_dict)
    avg_forecasted_vol_df = pd.DataFrame(avg_forecasted_vol_list, index=returns_df.columns, columns=['avg_vol'])

    return vol_forecast_df, avg_forecasted_vol_df
def egarch_forecast_series(tickers,returns):
    vol_forecast_dict = {}
    avg_forecasted_vol_list=[]
    for ticker in tqdm(tickers):
        window_size = 90
        forecast_horizon = 1  #one day ahead
        rolling_volatility = []
        dates = []

        for i in range(window_size, len(returns)):
            train_data = returns[ticker].iloc[i-window_size:i]

            if train_data.isna().any() or train_data.std() < 1e-6:
                continue  # skip bad segments

            model = arch_model(train_data, vol='EGARCH', p=1, q=1, o=1, dist='t')

            try:
                result = model.fit(disp='off', options={"maxiter": 5000},show_warning=False)
                forecast = result.forecast(horizon=forecast_horizon)
                var = forecast.variance.values[-1, 0]

                if not np.isnan(var) and var < 10000:
                    vol = np.sqrt(var) /100   # divide because train_data was scaled
                    rolling_volatility.append(vol)
                    dates.append(returns.index[i])
            except Exception as e:
                continue

        vol_series = pd.Series(rolling_volatility, index=dates)
        vol_forecast_dict[ticker]=vol_series

        avg_forecasted_vol=np.mean(rolling_volatility)
        avg_forecasted_vol_list.append(avg_forecasted_vol)
        print("avg_forecasted_vol of "+str(ticker)+" = "+str(avg_forecasted_vol))


    vol_forecast_df = pd.DataFrame(vol_forecast_dict)
    avg_forecasted_vol_df=pd.DataFrame(avg_forecasted_vol_list,index=tickers)
    return vol_forecast_df,avg_forecasted_vol_df

In [ ]:
vol_forecast_garch,avg_forecasted_vol_garch = garch_forecast_series(returns,tickers)

  2%|▏         | 1/48 [00:22<17:52, 22.83s/it]

avg_forecasted_vol of RELIANCE.NS = 19.72109918790152


  4%|▍         | 2/48 [00:40<15:15, 19.90s/it]

avg_forecasted_vol of TCS.NS = 20.618277046130395


  6%|▋         | 3/48 [00:58<14:12, 18.94s/it]

...

avg_forecasted_vol of M&M.NS = 25.592669719979042


 98%|█████████▊| 47/48 [14:12<00:17, 17.57s/it]

avg_forecasted_vol of INDUSINDBK.NS = 27.71629172463828


100%|██████████| 48/48 [14:31<00:00, 18.16s/it]

avg_forecasted_vol of APOLLOHOSP.NS = 23.948952656678877


In [ ]:
closing_data=yf.download(tickers,start=start_date,end=end_date,interval="1d")
closing_data=closing_data["Close"]
log_returns=np.log(closing_data/closing_data.shift(1))
expected_returns=log_returns.mean()
expected_returns_df=pd.DataFrame(expected_returns,columns=["expected returns"],index=tickers)
expected_returns
print(tickers)

[*********************100%***********************]  48 of 48 completed


['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'ITC.NS', 'KOTAKBANK.NS', 'LT.NS', 'SBIN.NS', 'AXISBANK.NS', 'BAJFINANCE.NS', 'BHARTIARTL.NS', 'ASIANPAINT.NS', 'HCLTECH.NS', 'MARUTI.NS', 'SUNPHARMA.NS', 'TITAN.NS', 'ULTRACEMCO.NS', 'WIPRO.NS', 'TECHM.NS', 'NESTLEIND.NS', 'POWERGRID.NS', 'NTPC.NS', 'JSWSTEEL.NS', 'TATASTEEL.NS', 'BAJAJFINSV.NS', 'ADANIENT.NS', 'ADANIPORTS.NS', 'HINDALCO.NS', 'ONGC.NS', 'COALINDIA.NS', 'EICHERMOT.NS', 'HDFCLIFE.NS', 'SBILIFE.NS', 'BRITANNIA.NS', 'DIVISLAB.NS', 'CIPLA.NS', 'GRASIM.NS', 'BPCL.NS', 'IOC.NS', 'HEROMOTOCO.NS', 'BAJAJ-AUTO.NS', 'SHREECEM.NS', 'DRREDDY.NS', 'M&M.NS', 'INDUSINDBK.NS', 'APOLLOHOSP.NS']


In [ ]:
log_returns

Ticker,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BHARTIARTL.NS,BPCL.NS,...,SBILIFE.NS,SBIN.NS,SHREECEM.NS,SUNPHARMA.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2022-03-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-03,0.001217,-0.005081,0.011566,-0.053242,-0.016937,-0.022552,-0.021974,-0.012253,-0.003268,0.012491,...,-0.036191,-0.014549,-0.047298,0.000000,0.010761,-0.000494,0.022901,-0.007122,-0.067659,0.025518
2022-03-04,-0.017676,-0.009312,-0.035296,-0.047691,-0.031187,-0.016822,-0.023902,-0.031682,-0.027684,0.005614,...,-0.022398,-0.011729,-0.012196,0.010603,-0.020308,-0.005673,0.018714,-0.053150,0.011570,0.010045
2022-03-07,-0.032056,-0.032443,-0.017617,-0.010906,-0.068668,-0.002301,-0.064655,-0.065237,0.032657,-0.030162,...,-0.027276,-0.048000,-0.025147,-0.008473,0.011522,-0.011257,-0.020503,-0.021240,-0.059975,-0.006015
2022-03-08,0.023725,0.016171,0.012337,0.005504,0.004260,0.025170,0.005378,0.005998,0.011190,0.010595,...,0.002960,0.000000,0.026632,0.038571,-0.017491,0.032481,0.026231,-0.007751,0.023476,0.026915
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-01-23,-0.006572,-0.013409,0.009650,-0.035740,-0.024754,-0.005712,-0.011310,-0.034528,0.028911,-0.037928,...,-0.046242,-0.036748,0.031188,0.031470,-0.031400,-0.021741,-0.016869,-0.018795,-0.026371,-0.031732
2024-01-24,0.002379,-0.014617,0.009638,-0.018253,-0.028023,0.016347,0.023945,-0.002180,0.027217,0.029996,...,0.025162,0.021580,0.005130,0.002753,0.038082,-0.004273,0.029772,0.007513,0.016094,0.019326
2024-01-25,-0.003398,0.022675,-0.007188,-0.016895,-0.016085,0.051955,0.005721,0.004271,-0.025017,-0.005780,...,-0.021244,-0.009017,0.011286,-0.010035,-0.010413,-0.008233,-0.062951,0.000623,-0.002119,-0.016983


In [ ]:
log_returns.to_pickle('log_returns_df.pkl')

In [ ]:
vol_forecast_garch.to_pickle("garch_vol_forecast.pkl")
avg_forecasted_vol_garch.to_pickle("garch_avg_vol_forecast.pkl")
expected_returns_df.to_pickle('expected_returns_df.pkl')
with open('tickers.pkl','wb') as f:
    pickle.dump(tickers,f)

NameError: name 'vol_forecast_garch' is not defined